<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/15-unsupervised-learning/02_exercises.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 15: Exercises

Twelve exercises: four Easy, five Medium and three Hard. You'll build the core of unsupervised learning yourself: the two steps of k-means and the whole algorithm, k-means++ starts, the silhouette, PCA through the SVD, the adjusted Rand index for comparing clusterings, robust z-scores, and finally DBSCAN, single-linkage hierarchical clustering and the EM algorithm for a Gaussian mixture.

Most checks compare your code with scikit-learn's version of the same method, so when they pass, you know your version computes the same thing.

**Run the setup cell first.**

Rules:

- NumPy and pandas. Don't call scikit-learn's version of what you're writing; the checks use it to test your answer.
- Solutions are in `03_solutions.ipynb`.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from sklearn.cluster import DBSCAN, AgglomerativeClustering, KMeans
from sklearn.datasets import make_blobs, make_moons
from sklearn.decomposition import PCA
from sklearn.metrics import adjusted_rand_score, rand_score, silhouette_samples
from sklearn.mixture import GaussianMixture


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _blobs(seed=0, n=300, centers=4, std=1.0):
    X, y = make_blobs(n_samples=n, centers=centers, cluster_std=std, random_state=seed)
    return X, y


def check_assign_to_nearest(f):
    rng = np.random.default_rng(0)
    X = rng.normal(size=(500, 3))
    C = rng.normal(size=(6, 3))
    reference = ((X[:, None, :] - C[None]) ** 2).sum(axis=2).argmin(axis=1)
    _scenarios("assign_to_nearest", [
        ("the example", lambda: f(np.array([[0, 0], [9, 9], [4, 4]]), np.array([[1, 1], [8, 8]])).tolist(), [0, 1, 0],
         "squared distance from every point to every centroid, then argmin"),
        ("ties go to the lower index", lambda: f(np.array([[0.0, 0.0]]), np.array([[1.0, 0.0], [-1.0, 0.0]])).tolist(), [0], None),
        ("random data in 3 dimensions", lambda: f(X, C).tolist(), reference.tolist(), None),
        ("returns integers", lambda: f(X, C).dtype.kind, "i", None),
    ])


def check_inertia(f):
    X, _ = _blobs(1)
    km = KMeans(n_clusters=4, n_init=5, random_state=0).fit(X)
    _scenarios("inertia", [
        ("a small example", lambda: f(np.array([[0, 0], [2, 0], [10, 10]]), np.array([0, 0, 1]), np.array([[1, 0], [10, 10]])), 2.0,
         "sum of squared distances to each point's own centroid"),
        ("matches scikit-learn's inertia_", lambda: f(X, km.labels_, km.cluster_centers_), float(km.inertia_), None),
        ("zero when every point is a centroid", lambda: f(X[:5], np.arange(5), X[:5]), 0.0, None),
        ("returns a float", lambda: type(f(X, km.labels_, km.cluster_centers_)).__name__, "float", None),
    ])


def check_robust_z_scores(f):
    rng = np.random.default_rng(2)
    x = np.r_[rng.normal(50, 5, 200), 500.0]
    med = np.median(x)
    mad = np.median(np.abs(x - med))
    _scenarios("robust_z_scores", [
        ("a small example", lambda: f([1, 2, 3, 4, 100]), (np.array([1, 2, 3, 4, 100]) - 3) / (1.4826 * 1.0),
         "median 3, MAD = median of |x - 3| = 1"),
        ("matches the formula", lambda: f(x), (x - med) / (1.4826 * mad), None),
        ("the outlier stands out far more than with ordinary z-scores",
         lambda: bool(f(x)[-1] > 3 * ((x[-1] - x.mean()) / x.std())), True, None),
        ("normal data: about the same as ordinary z-scores", lambda: abs(np.std(f(rng.normal(0, 1, 20000))) - 1) < 0.05, True, None),
        ("MAD of 0 raises", lambda: f([5, 5, 5, 5, 9]), Raises(ValueError), "check whether the MAD is 0"),
    ])


def check_cluster_profile(f):
    df = pd.DataFrame({"spend": [10.0, 12, 11, 100, 120, 5000], "visits": [1, 2, 3, 10, 11, 12]}, index=[10, 3, 7, 1, 4, 99])
    labels = [1, 1, 1, 0, 0, 0]
    expected = pd.DataFrame({"size": [3, 3], "share": [0.5, 0.5], "spend": [120.0, 11.0], "visits": [11.0, 2.0]}, index=[0, 1])
    before = df.copy()

    def unchanged():
        f(df, labels)
        return bool(df.equals(before))

    def strip(t):
        out = t.copy()
        out.index = list(out.index)
        out.index.name = None
        return out

    _scenarios("cluster_profile", [
        ("a small example, with a shuffled index", lambda: strip(f(df, labels)), expected,
         "one row per cluster: size, share, then the median of each column"),
        ("sorted by cluster label", lambda: list(f(df, [2, 0, 1, 2, 0, 1]).index), [0, 1, 2], None),
        ("size is an int", lambda: f(df, labels)["size"].dtype.kind, "i", None),
        ("df is not changed", unchanged, True, "don't add a column to df"),
    ])


def check_kmeans(f):
    X, _ = _blobs(3, n=400, centers=5, std=1.5)
    rng = np.random.default_rng(3)
    init = X[rng.choice(len(X), 5, replace=False)]
    sk = KMeans(n_clusters=5, init=init, n_init=1, algorithm="lloyd", max_iter=300, tol=0).fit(X)
    init_copy = init.copy()

    def untouched():
        f(X, init)
        return bool(np.array_equal(init, init_copy))

    def inertia_never_rises():
        values = []
        for steps in range(1, 8):
            labels, centroids, _ = f(X, init, max_iter=steps)
            values.append(((X - centroids[labels]) ** 2).sum())
        return bool(all(b <= a + 1e-9 for a, b in zip(values, values[1:])))

    _scenarios("kmeans", [
        ("a tiny example", lambda: tuple(np.asarray(v).tolist() for v in f(np.array([[0.0], [1.0], [10.0], [11.0]]), np.array([[0.0], [1.0]]))[:2]),
         ([0, 0, 1, 1], [[0.5], [10.5]]), "assign, update, repeat until no label changes"),
        ("same labels as scikit-learn", lambda: adjusted_rand_score(sk.labels_, f(X, init)[0]), 1.0, None),
        ("same centroids as scikit-learn", lambda: np.round(np.sort(f(X, init)[1], axis=0), 6),
         np.round(np.sort(sk.cluster_centers_, axis=0), 6), None),
        ("max_iter is respected", lambda: f(X, init, max_iter=2)[2], 2, None),
        ("an empty cluster keeps its centroid", lambda: f(np.array([[0.0], [1.0]]), np.array([[0.5], [100.0]]))[1].tolist(), [[0.5], [100.0]],
         "skip clusters with no points"),
        ("the inertia never goes up", inertia_never_rises, True, None),
        ("the starting centroids are not changed", untouched, True, "work on a copy"),
    ])


def check_kmeans_plus_plus(f):
    X, y = _blobs(4, n=1000, centers=10, std=0.4)

    def reference(Xr, k, seed):
        g = np.random.default_rng(seed)
        c = [Xr[g.integers(len(Xr))]]
        d2 = ((Xr - c[0]) ** 2).sum(axis=1)
        while len(c) < k:
            nxt = Xr[g.choice(len(Xr), p=d2 / d2.sum())]
            c.append(nxt)
            d2 = np.minimum(d2, ((Xr - nxt) ** 2).sum(axis=1))
        return np.array(c)

    def groups_found(starts):
        return len(set(y[((X[:, None, :] - starts[None]) ** 2).sum(axis=2).argmin(axis=0)]))

    def plus_plus_spreads_out():
        pp = np.mean([groups_found(f(X, 10, seed=s)) for s in range(20)])
        random_starts = np.mean([groups_found(X[np.random.default_rng(s).choice(len(X), 10, replace=False)]) for s in range(20)])
        return bool(pp > random_starts + 1)

    _scenarios("kmeans_plus_plus", [
        ("matches the reference procedure", lambda: f(X, 5, seed=0), reference(X, 5, 0), "one rng: integers for the first, choice with p for the rest"),
        ("the seed is used", lambda: f(X, 3, seed=7), reference(X, 3, 7), None),
        ("returns k distinct points from X", lambda: (f(X, 10).shape, len({tuple(r) for r in f(X, 10)})), ((10, 2), 10), None),
        ("spreads out better than random starts", plus_plus_spreads_out, True, None),
        ("k = 0 raises", lambda: f(X, 0), Raises(ValueError), None),
        ("k larger than n raises", lambda: f(X[:3], 4), Raises(ValueError), None),
    ])


def check_silhouette_samples_manual(f):
    X, _ = _blobs(5, n=200, centers=3, std=1.2)
    labels = KMeans(n_clusters=3, n_init=5, random_state=0).fit_predict(X)
    wrong = labels.copy()
    far_point = int(np.argmax(X[:, 0]))
    wrong[far_point] = (labels[far_point] + 1) % 3
    tight, tight_y = _blobs(6, n=150, centers=[[0, 0], [20, 20], [-20, 20]], std=0.5)
    single = np.r_[np.zeros(5, dtype=int), 1]
    X_single = np.r_[np.random.default_rng(0).normal(size=(5, 2)), [[10.0, 10.0]]]
    _scenarios("silhouette_samples_manual", [
        ("matches scikit-learn", lambda: f(X, labels), silhouette_samples(X, labels), None),
        ("a point in the wrong cluster is negative", lambda: bool(f(X, wrong)[far_point] < 0), True, None),
        ("matches scikit-learn with a misplaced point", lambda: f(X, wrong), silhouette_samples(X, wrong), "divide by max(a, b)"),
        ("well-separated blobs average near 1", lambda: bool(f(tight, tight_y).mean() > 0.9), True, None),
        ("a point alone in its cluster gets 0", lambda: float(f(X_single, single)[-1]), 0.0, "singletons get 0"),
        ("string labels work", lambda: f(X, np.array(["a", "b", "c"])[labels]), silhouette_samples(X, labels), None),
        ("one cluster raises", lambda: f(X, np.zeros(len(X))), Raises(ValueError), None),
    ])


def check_pca_svd(f):
    rng = np.random.default_rng(7)
    X = rng.normal(size=(300, 5)) @ rng.normal(size=(5, 5)) + 100       # correlated, and far from the origin
    sk = PCA(n_components=3).fit(X)
    result = {}

    def run():
        if "v" not in result:
            result["v"] = f(X, 3)
        return result["v"]

    def scores_uncorrelated():
        s = run()["scores"]
        cov = np.cov(s, rowvar=False)
        return bool(np.allclose(cov - np.diag(np.diag(cov)), 0, atol=1e-8) and np.allclose(np.diag(cov), run()["explained_variance"]))

    _scenarios("pca_svd", [
        ("mean", lambda: run()["mean"], X.mean(axis=0), None),
        ("components match scikit-learn", lambda: run()["components"], sk.components_,
         "centre first; flip each component so its largest-magnitude entry is positive"),
        ("explained variance", lambda: run()["explained_variance"], sk.explained_variance_, "S² / (n - 1)"),
        ("explained variance ratio", lambda: run()["explained_variance_ratio"], sk.explained_variance_ratio_,
         "divide by the total variance of all directions"),
        ("scores match transform", lambda: run()["scores"], sk.transform(X), None),
        ("scores are uncorrelated, with the explained variances", scores_uncorrelated, True, None),
    ])


def check_adjusted_rand_index(f):
    rng = np.random.default_rng(8)
    a = rng.integers(0, 5, 500)
    b = np.where(rng.random(500) < 0.7, a, rng.integers(0, 5, 500))
    r1 = rng.integers(0, 10, 1000)
    r2 = rng.integers(0, 10, 1000)
    _scenarios("adjusted_rand_index", [
        ("identical clusterings give 1", lambda: f([0, 0, 1, 1, 2], [0, 0, 1, 1, 2]), 1.0, None),
        ("renamed clusters still give 1", lambda: f([0, 0, 1, 1, 2], ["x", "x", "z", "z", "y"]), 1.0, "only the grouping matters"),
        ("matches scikit-learn", lambda: f(a, b), float(adjusted_rand_score(a, b)), None),
        ("different numbers of clusters", lambda: f(a, a // 2), float(adjusted_rand_score(a, a // 2)), None),
        ("unrelated clusterings: about 0, while the plain Rand index is high",
         lambda: (abs(f(r1, r2)) < 0.02, rand_score(r1, r2) > 0.8), (True, True), None),
        ("everything in one cluster in both gives 1", lambda: f([1, 1, 1], [5, 5, 5]), 1.0, "if the denominator is 0, return 1.0"),
        ("returns a float", lambda: type(f(a, b)).__name__, "float", None),
    ])


def check_dbscan(f):
    rng = np.random.default_rng(9)
    X_m, _ = make_moons(n_samples=300, noise=0.06, random_state=0)
    X = np.vstack([X_m, rng.uniform([-1.5, -1], [2.5, 1.5], size=(30, 2))])
    eps, ms = 0.15, 5
    sk = DBSCAN(eps=eps, min_samples=ms).fit(X)
    core = np.zeros(len(X), dtype=bool)
    core[sk.core_sample_indices_] = True
    D = np.sqrt(((X[:, None, :] - X[None]) ** 2).sum(axis=2))
    result = {}

    def run():
        if "v" not in result:
            result["v"] = np.asarray(f(X, eps, ms))
        return result["v"]

    def borders_ok():
        labels = run()
        for i in np.flatnonzero((labels != -1) & ~core):
            near_core = np.flatnonzero((D[i] <= eps) & core)
            if not np.any(labels[near_core] == labels[i]):
                return False
        return True

    _scenarios("dbscan", [
        ("a small example", lambda: f(np.array([[0.0], [0.1], [0.2], [5.0], [5.1], [5.2], [9.0]]), 0.15, 2).tolist(),
         [0, 0, 0, 1, 1, 1, -1], "neighbourhoods include the point itself"),
        ("a distance exactly eps counts as a neighbour", lambda: f(np.array([[0.0], [1.0], [2.0]]), 1.0, 2).tolist(), [0, 0, 0], "use <= eps"),
        ("the point itself counts towards min_samples", lambda: f(np.array([[0.0], [0.1], [5.0]]), 0.15, 2).tolist(), [0, 0, -1], None),
        ("the same noise points as scikit-learn", lambda: np.flatnonzero(run() == -1).tolist(), np.flatnonzero(sk.labels_ == -1).tolist(), None),
        ("the same clusters of core points (ARI)", lambda: float(adjusted_rand_score(sk.labels_[core], run()[core])), 1.0, None),
        ("clusters numbered from 0 in order of discovery", lambda: sorted(set(run().tolist()) - {-1}), list(range(len(set(sk.labels_) - {-1}))), None),
        ("every border point is next to a core point of its cluster", borders_ok, True, "only explore the neighbourhoods of core points"),
        ("min_samples too high: everything is noise", lambda: set(f(X, 0.05, 50).tolist()), {-1}, None),
    ])


def check_single_linkage_clusters(f):
    X_m, y_m = make_moons(n_samples=200, noise=0.05, random_state=1)
    rng = np.random.default_rng(10)
    X_bridge = np.vstack([X_m, np.column_stack([np.linspace(-0.2, 1.2, 25), np.full(25, 0.25)]) + rng.normal(0, 0.01, (25, 2))])
    X_b, _ = _blobs(11, n=120, centers=3, std=1.0)
    _scenarios("single_linkage_clusters", [
        ("a small example", lambda: f(np.array([[0.0], [0.5], [5.0], [5.4], [20.0]]), 3).tolist(), [0, 0, 1, 1, 2],
         "merge the closest pair of clusters, again and again"),
        ("labels numbered by smallest point index", lambda: f(np.array([[9.0], [0.0], [0.1], [9.1]]), 2).tolist(), [0, 1, 1, 0], None),
        ("matches scikit-learn on blobs", lambda: float(adjusted_rand_score(
            AgglomerativeClustering(n_clusters=3, linkage="single").fit_predict(X_b), f(X_b, 3))), 1.0, None),
        ("separates two clean moons", lambda: float(adjusted_rand_score(y_m, f(X_m, 2))), 1.0, None),
        ("matches scikit-learn when a bridge joins the moons", lambda: float(adjusted_rand_score(
            AgglomerativeClustering(n_clusters=2, linkage="single").fit_predict(X_bridge), f(X_bridge, 2))), 1.0, None),
        ("n_clusters = n gives every point its own cluster", lambda: f(X_b[:6], 6).tolist(), list(range(6)), None),
    ])


def check_gaussian_mixture_1d(f):
    rng = np.random.default_rng(12)
    x = np.r_[rng.normal(-2, 0.7, 600), rng.normal(3, 1.5, 400)]

    def reference(xx, mu, var, w, n_iter):
        mu, var, w = np.array(mu, float), np.array(var, float), np.array(w, float)
        hist = []
        for _ in range(n_iter + 1):
            dens = w * np.exp(-(xx[:, None] - mu) ** 2 / (2 * var)) / np.sqrt(2 * np.pi * var)
            tot = dens.sum(axis=1)
            hist.append(float(np.log(tot).sum()))
            if len(hist) == n_iter + 1:
                break
            r = dens / tot[:, None]
            Nk = r.sum(axis=0)
            w = Nk / len(xx)
            mu = (r * xx[:, None]).sum(axis=0) / Nk
            var = (r * (xx[:, None] - mu) ** 2).sum(axis=0) / Nk
        return {"means": mu, "variances": var, "weights": w, "log_likelihood": hist}

    start = ([-1.0, 1.0], [1.0, 1.0], [0.5, 0.5])
    result = {}

    def run():
        if "v" not in result:
            result["v"] = f(x, *start, n_iter=50)
        return result["v"]

    def close_to_sklearn():
        gm = GaussianMixture(2, means_init=[[-1.0], [1.0]], precisions_init=[[[1.0]], [[1.0]]], weights_init=[0.5, 0.5],
                             reg_covar=0, tol=1e-12, max_iter=500).fit(x[:, None])
        order = np.argsort(gm.means_[:, 0])
        ours = f(x, *start, n_iter=500)
        return bool(np.allclose(ours["means"], gm.means_[order, 0], atol=1e-3) and np.allclose(ours["weights"], gm.weights_[order], atol=1e-3))

    _scenarios("gaussian_mixture_1d", [
        ("one iteration matches the reference", lambda: f(x, *start, n_iter=1), reference(x, *start, 1),
         "E-step: responsibilities; M-step: weights, then means, then variances with the new means"),
        ("fifty iterations match the reference", run, reference(x, *start, 50), None),
        ("n_iter + 1 log-likelihood values", lambda: len(run()["log_likelihood"]), 51, None),
        ("the log-likelihood never decreases", lambda: bool(np.all(np.diff(run()["log_likelihood"]) >= -1e-9)), True, None),
        ("recovers the true means (-2, 3) and weights (0.6, 0.4)",
         lambda: (np.round(run()["means"], 0).tolist(), np.round(run()["weights"], 1).tolist()), ([-2.0, 3.0], [0.6, 0.4]), None),
        ("matches scikit-learn's GaussianMixture from the same start", close_to_sklearn, True, None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `assign_to_nearest`

Write `assign_to_nearest(X, centroids)`: the assignment step of k-means. Return an integer array with, for each row of `X`, the index of the nearest centroid (by ordinary Euclidean distance).

- `X` has shape `(n, d)` and `centroids` has shape `(k, d)`.
- If a point is equally close to two centroids, choose the lower index.
- Use NumPy broadcasting, not a Python loop over the points.

Example: `assign_to_nearest(np.array([[0, 0], [9, 9], [4, 4]]), np.array([[1, 1], [8, 8]]))` returns `array([0, 1, 0])`.

In [2]:
def assign_to_nearest(X, centroids) -> np.ndarray:
    # your code here
    pass

In [3]:
check_assign_to_nearest(assign_to_nearest)

FAIL  assign_to_nearest: the example
      raised AttributeError: 'NoneType' object has no attribute 'tolist'
      expected: [0, 1, 0]
      hint: squared distance from every point to every centroid, then argmin


### Exercise 2: `inertia`

Write `inertia(X, labels, centroids)` that returns the total squared distance from each point to the centroid of its cluster, as a float. This is the quantity k-means minimises.

- `labels[i]` is the cluster index of row `i`, so its centroid is `centroids[labels[i]]`.

In [4]:
def inertia(X, labels, centroids) -> float:
    # your code here
    pass

In [5]:
check_inertia(inertia)

FAIL  inertia: a small example
      expected: 2.0
      got:      None
      hint: sum of squared distances to each point's own centroid


### Exercise 3: `robust_z_scores`

Write `robust_z_scores(x)` that returns a robust z-score for each value of a 1D array:

$$z_i = \frac{x_i - \text{median}(x)}{1.4826 \cdot \text{MAD}}, \qquad \text{MAD} = \text{median}(|x - \text{median}(x)|)$$

- Return a float array.
- Raise `ValueError` if the MAD is 0 (for example, if more than half the values are identical), since the scores would be undefined.

The factor 1.4826 makes the MAD match the standard deviation for normally distributed data, so the scores can be read like ordinary z-scores.

In [6]:
def robust_z_scores(x) -> np.ndarray:
    # your code here
    pass

In [7]:
check_robust_z_scores(robust_z_scores)

FAIL  robust_z_scores: a small example
      expected: array(shape=(5,), dtype=float64, [-1.349 , -0.6745, 0. , 0.6745, 65.4256])
      got:      None
      hint: median 3, MAD = median of |x - 3| = 1


### Exercise 4: `cluster_profile`

Write `cluster_profile(df, labels)` that summarises clusters in a table a person can read. `df` is a DataFrame of numeric features and `labels` the cluster of each row.

Return a DataFrame with one row per cluster, sorted by cluster label, containing:

- a column `"size"`: the number of rows in the cluster (an int)
- a column `"share"`: the cluster's fraction of all rows
- then the **median** of every column of `df`, in the same order as `df`

Don't modify `df`.

In [8]:
def cluster_profile(df: pd.DataFrame, labels) -> pd.DataFrame:
    # your code here
    pass

In [9]:
check_cluster_profile(cluster_profile)

FAIL  cluster_profile: a small example, with a shuffled index
      raised AttributeError: 'NoneType' object has no attribute 'copy'
      expected: DataFrame shape=(2, 4)
             size  share  spend  visits
          0     3    0.5  120.0    11.0
          1     3    0.5   11.0     2.0
      hint: one row per cluster: size, share, then the median of each column


## Medium

### Exercise 5: `kmeans`

Write `kmeans(X, initial_centroids, max_iter=300)` using Lloyd's algorithm. Return `(labels, centroids, n_iter)`.

Starting from the given centroids, repeat up to `max_iter` times:

1. **Assign** each point to its nearest centroid (your `assign_to_nearest` is available).
2. **Update** each centroid to the mean of its points. If a cluster has no points, keep its old centroid.
3. If no point changed cluster since the previous assignment, stop.

`n_iter` is the number of assignment steps carried out. Return the final labels as an int array and the final centroids as a float array.

In [10]:
def kmeans(X, initial_centroids, max_iter: int = 300):
    # your code here
    pass

In [11]:
check_kmeans(kmeans)

FAIL  kmeans: a tiny example
      raised TypeError: 'NoneType' object is not subscriptable
      expected: ([0, 0, 1, 1], [[0.5], [10.5]])
      hint: assign, update, repeat until no label changes


### Exercise 6: `kmeans_plus_plus`

Write `kmeans_plus_plus(X, k, seed=0)` that picks `k` starting centroids with the k-means++ method. Return them as a `(k, d)` float array.

Follow this procedure exactly:

```
rng = np.random.default_rng(seed)
first centroid: X[rng.integers(len(X))]
repeat until there are k centroids:
    d2 = squared distance from each point to its nearest chosen centroid
    next centroid: X[rng.choice(len(X), p=d2 / d2.sum())]
```

Raise `ValueError` if `k` is less than 1 or greater than the number of rows.

In [12]:
def kmeans_plus_plus(X, k: int, seed: int = 0) -> np.ndarray:
    # your code here
    pass

In [13]:
check_kmeans_plus_plus(kmeans_plus_plus)

FAIL  kmeans_plus_plus: matches the reference procedure
      expected: array(shape=(5, 2), dtype=float64, [[ 9.5075, 5.0441], [-9.8744, -2.3641], [ 5.3225, -5...
      got:      None
      hint: one rng: integers for the first, choice with p for the rest


### Exercise 7: `silhouette_samples_manual`

Write `silhouette_samples_manual(X, labels)` that returns the silhouette of every point as a float array:

$$s_i = \frac{b_i - a_i}{\max(a_i, b_i)}$$

- `a_i`: the average distance from point `i` to the **other** points in its own cluster
- `b_i`: the smallest, over the other clusters, of the average distance from point `i` to that cluster's points
- A point that is alone in its cluster has `s_i = 0`.
- Raise `ValueError` unless there are at least 2 clusters and fewer clusters than points.

Use Euclidean distances. The check compares with `sklearn.metrics.silhouette_samples`; don't call it yourself.

In [14]:
def silhouette_samples_manual(X, labels) -> np.ndarray:
    # your code here
    pass

In [15]:
check_silhouette_samples_manual(silhouette_samples_manual)

FAIL  silhouette_samples_manual: matches scikit-learn
      expected: array(shape=(200,), dtype=float64, [0.4767, 0.4095, 0.3494, ..., 0.8026, 0.3739, 0.6854])
      got:      None


### Exercise 8: `pca_svd`

Write `pca_svd(X, n_components)` that performs PCA through the singular value decomposition. Return a dict with:

- `"mean"`: the mean of each column (shape `(d,)`)
- `"components"`: the first `n_components` principal directions, as rows (shape `(n_components, d)`)
- `"explained_variance"`: the variance along each of those directions, `S² / (n - 1)`
- `"explained_variance_ratio"`: each variance as a share of the **total** variance of all directions
- `"scores"`: the centred data projected onto the components (shape `(n, n_components)`)

**Sign convention**: flip the sign of any component whose entry with the largest absolute value is negative (and flip the matching column of scores), so the largest entry of every component is positive. That makes results unique.

In [16]:
def pca_svd(X, n_components: int) -> dict:
    # your code here
    pass

In [17]:
check_pca_svd(pca_svd)

FAIL  pca_svd: mean
      raised TypeError: 'NoneType' object is not subscriptable
      expected: array(shape=(5,), dtype=float64, [100.2012, 99.5636, 99.9889, 100.111 , 99.5539])


### Exercise 9: `adjusted_rand_index`

Write `adjusted_rand_index(labels_a, labels_b)` that compares two clusterings of the same points and returns the adjusted Rand index as a float.

Build the **contingency table** `n_ij` (how many points are in cluster `i` of `a` and cluster `j` of `b`), with row sums `a_i` and column sums `b_j`. With `C(m) = m(m - 1)/2`, the number of pairs among `m` items:

$$\text{ARI} = \frac{\sum_{ij} C(n_{ij}) - E}{\tfrac{1}{2}\left[\sum_i C(a_i) + \sum_j C(b_j)\right] - E}, \qquad E = \frac{\sum_i C(a_i) \sum_j C(b_j)}{C(n)}$$

- Labels can be any values, and the two clusterings can have different numbers of clusters.
- If the denominator is 0 (for example, both clusterings put everything in one cluster), return `1.0`.

Don't use scikit-learn's metric functions.

In [18]:
def adjusted_rand_index(labels_a, labels_b) -> float:
    # your code here
    pass

In [19]:
check_adjusted_rand_index(adjusted_rand_index)

FAIL  adjusted_rand_index: identical clusterings give 1
      expected: 1.0
      got:      None


## Hard

### Exercise 10: `dbscan`

Write `dbscan(X, eps, min_samples)` from scratch. Return an int array of cluster labels, with `-1` for noise.

- A point's **neighbourhood** is every point within distance `eps` of it (`<= eps`), **including itself**. A **core point** has at least `min_samples` points in its neighbourhood.
- Go through the points in index order. When you reach a core point that has no cluster yet, start a new cluster (numbered 0, 1, 2, ... in the order they're started) and grow it: every point in a core point's neighbourhood joins the cluster, and the neighbourhoods of the **core** points among them are explored in turn (a queue or a stack works).
- A non-core point that joins a cluster is a **border** point; it stays in the first cluster that reaches it, and its own neighbourhood isn't explored.
- Points never reached are noise.

A pairwise distance matrix is fine for the sizes in the check.

In [20]:
def dbscan(X, eps: float, min_samples: int) -> np.ndarray:
    # your code here
    pass

In [21]:
check_dbscan(dbscan)

FAIL  dbscan: a small example
      raised AttributeError: 'NoneType' object has no attribute 'tolist'
      expected: [0, 0, 0, 1, 1, 1, -1]
      hint: neighbourhoods include the point itself


### Exercise 11: `single_linkage_clusters`

Write `single_linkage_clusters(X, n_clusters)` that performs agglomerative clustering with **single linkage** and returns an int array of cluster labels.

- Start with every point as its own cluster.
- Repeatedly merge the two clusters whose **closest pair of points** is nearest, until `n_clusters` clusters remain.
- Number the final clusters 0, 1, 2, ... in order of their **smallest point index** (the cluster containing point 0 is cluster 0, and so on).

The check uses a few hundred points; anything up to `n²` memory is fine, but avoid recomputing all cluster distances from scratch after every merge.

In [22]:
def single_linkage_clusters(X, n_clusters: int) -> np.ndarray:
    # your code here
    pass

In [23]:
check_single_linkage_clusters(single_linkage_clusters)

FAIL  single_linkage_clusters: a small example
      raised AttributeError: 'NoneType' object has no attribute 'tolist'
      expected: [0, 0, 1, 1, 2]
      hint: merge the closest pair of clusters, again and again


### Exercise 12: `gaussian_mixture_1d`

Write `gaussian_mixture_1d(x, means, variances, weights, n_iter)` that fits a mixture of 1D normal distributions with the **EM algorithm**, starting from the given parameters (each a list or array with one entry per component). Return a dict with the final `"means"`, `"variances"` and `"weights"` (float arrays) and `"log_likelihood"`: a list with the log-likelihood of the data **before** each of the `n_iter` iterations, plus once more at the end (so `n_iter + 1` values).

One iteration:

- **E-step**: for each point `i` and component `k`, the responsibility `r_ik = w_k N(x_i | μ_k, σ²_k) / Σ_j w_j N(x_i | μ_j, σ²_j)`.
- **M-step**: with `N_k = Σ_i r_ik`: `w_k = N_k / n`, `μ_k = Σ_i r_ik x_i / N_k`, and `σ²_k = Σ_i r_ik (x_i - μ_k)² / N_k`, using the **new** `μ_k`.

The log-likelihood is `Σ_i log Σ_k w_k N(x_i | μ_k, σ²_k)`, where `N(x | μ, σ²) = exp(-(x - μ)² / (2σ²)) / sqrt(2πσ²)`.

In [24]:
def gaussian_mixture_1d(x, means, variances, weights, n_iter: int) -> dict:
    # your code here
    pass

In [25]:
check_gaussian_mixture_1d(gaussian_mixture_1d)

FAIL  gaussian_mixture_1d: one iteration matches the reference
      expected: {'means': array([-1.94675964,  2.86064061]), 'variances': array([0.63135977, 2.97342883...
      got:      None
      hint: E-step: responsibilities; M-step: weights, then means, then variances with the new means


## Done?

Once everything passes, compare with `03_solutions.ipynb`. Then open the mini project in `04_mini_project.ipynb`, where these tools look for customer groups in the spending records of a real wholesale distributor.